In [1]:
import pandas as pd
import numpy as np

In [8]:
kaggle_quali = pd.read_csv('filler-datasets/qualifying.csv')
kaggle_quali = kaggle_quali[kaggle_quali['raceId']>1009]
kaggle_quali = kaggle_quali[kaggle_quali['raceId']<1074]
kaggle_quali.head()

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
7936,7960,1010,1,131,44,1,1:22.043,1:21.014,1:20.486
7937,7961,1010,822,131,77,2,1:22.367,1:21.193,1:20.598
7938,7962,1010,20,6,5,3,1:22.885,1:21.912,1:21.190
7939,7963,1010,830,9,33,4,1:22.876,1:21.678,1:21.320
7940,7964,1010,844,6,16,5,1:22.017,1:21.739,1:21.442


In [7]:
kaggle_quali.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1197 entries, 7936 to 9134
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   qualifyId      1197 non-null   int64 
 1   raceId         1197 non-null   int64 
 2   driverId       1197 non-null   int64 
 3   constructorId  1197 non-null   int64 
 4   number         1197 non-null   int64 
 5   position       1197 non-null   int64 
 6   q1             1197 non-null   object
 7   q2             1197 non-null   object
 8   q3             1197 non-null   object
dtypes: int64(6), object(3)
memory usage: 93.5+ KB


In [3]:
quali_2024 = pd.read_csv('quali_both_result(processed)/race_quali_2022.csv')
quali_2024.head()

,Unnamed: 0,raceId,Track,Team,kaggle_driver_id,Position,participated_q2,participated_q3,q1_time_sec,q2_time_sec,q3_time_sec
0,0,1074,Bahrain,Ferrari,844,1,1,1,91.471,90.932,90.558
1,1,1074,Bahrain,Red Bull Racing RBPT,830,2,1,1,91.785,90.757,90.681
2,2,1074,Bahrain,Ferrari,832,3,1,1,91.567,90.787,90.687
3,3,1074,Bahrain,Red Bull Racing RBPT,815,4,1,1,92.311,91.008,90.921
4,4,1074,Bahrain,Mercedes,1,5,1,1,92.285,91.048,91.238


In [11]:
kaggle_driver = pd.read_csv('filler-datasets/drivers_filtered.csv')

In [12]:
def normalize_name_series(s):
    return (
        s.str.lower()
         .str.strip()
         .str.normalize('NFKD')
         .str.encode('ascii', errors='ignore')
         .str.decode('utf-8')
    )

def add_kaggle_driver_id(df, kaggle_driver):
    df = df.copy()

    df['driver_norm'] = normalize_name_series(df['Driver'])

    kd = kaggle_driver.copy()
    kd['driver_norm'] = normalize_name_series(
        kd['forename'] + ' ' + kd['surname']
    )

    df = df.merge(
        kd[['driverId', 'driver_norm']],
        on='driver_norm',
        how='left'
    )

    df.rename(columns={'driverId': 'kaggle_driver_id'}, inplace=True)
    df.drop(columns=['driver_norm'], inplace=True)

    return df

In [13]:
season_2019 = pd.read_csv('formula1-datasets-master/formula1-datasets-master/formula1_2019season_drivers.csv')
season_2020 = pd.read_csv('formula1-datasets-master/formula1-datasets-master/formula1_2020season_drivers.csv')
season_2021 = pd.read_csv('formula1-datasets-master/formula1-datasets-master/formula1_2021season_drivers.csv')
season_2022 = pd.read_csv('formula1-datasets-master/formula1-datasets-master/formula1_2022season_drivers.csv')
season_2023 = pd.read_csv('formula1-datasets-master/formula1-datasets-master/formula1_2023season_drivers.csv')
season_2024 = pd.read_csv('formula1-datasets-master/formula1-datasets-master/formula1_2024season_drivers.csv')
season_2025 = pd.read_csv('formula1-datasets-master/formula1-datasets-master/formula1_2025Season_drivers.csv')


In [14]:
season_2019.head()

,Driver,Number,Team,Country,Podiums,Points,Grands Prix Entered,World Championships,Highest Race Finish,Highest Grid Position,Date of Birth,Place of Birth
0,Lewis Hamilton,44,Mercedes,United Kingdom,151,3431,250,6,1(x84),1,07/07/1985,"Stevenage, England"
1,Valtteri Bottas,77,Mercedes,Finland,45,1289,140,0,1(x7),1,28/08/1989,"Nastola, Finland"
2,Max Verstappen,33,Red Bull Racing,Netherlands,31,948,102,0,1(x8),1,30/09/1997,"Hasselt, Belgium"
3,Charles Leclerc,16,Ferrari,Monaco,10,303,42,0,1(x2),1,16/10/1997,"Monte Carlo, Monaco"
4,Sebastian Vettel,5,Ferrari,Germany,120,2985,241,4,1(x53),1,03/07/1987,"Heppenheim, Germany"


In [15]:
def process_season_drivers(df, year, kaggle_driver):
    df = df.copy()

    # Keep only required columns
    df = df[['Driver', 'Team']]

    # Add kaggle_driver_id
    df = add_kaggle_driver_id(df, kaggle_driver)

    # Reorder columns
    df = df[['kaggle_driver_id', 'Driver', 'Team']]

    # Save to root folder
    filename = f"f1{year}season_drivers_processed.csv"
    df.to_csv(filename, index=False)

    print(f"Saved: {filename}")

    return df

In [16]:
processed_2019 = process_season_drivers(season_2019, 2019, kaggle_driver)
processed_2020 = process_season_drivers(season_2020, 2020, kaggle_driver)
processed_2021 = process_season_drivers(season_2021, 2021, kaggle_driver)
processed_2022 = process_season_drivers(season_2022, 2022, kaggle_driver)
processed_2023 = process_season_drivers(season_2023, 2023, kaggle_driver)
processed_2024 = process_season_drivers(season_2024, 2024, kaggle_driver)
processed_2025 = process_season_drivers(season_2025, 2025, kaggle_driver)

Saved: f12019season_drivers_processed.csv
Saved: f12020season_drivers_processed.csv
Saved: f12021season_drivers_processed.csv
Saved: f12022season_drivers_processed.csv
Saved: f12023season_drivers_processed.csv
Saved: f12024season_drivers_processed.csv
Saved: f12025season_drivers_processed.csv


In [32]:
main_race_res_2019 = pd.read_csv('main_race_result(processed)/formula1_2019season_raceResults.csv')
main_race_res_2020 = pd.read_csv('main_race_result(processed)/formula1_2020season_raceResults.csv')
main_race_res_2021 = pd.read_csv('main_race_result(processed)/formula1_2021season_raceResults.csv')
main_race_res_2022 = pd.read_csv('main_race_result(processed)/formula1_2022season_raceResults.csv')
main_race_res_2023 = pd.read_csv('main_race_result(processed)/formula1_2023season_raceResults.csv')
main_race_res_2024 = pd.read_csv('main_race_result(processed)/formula1_2024season_raceResults.csv')
main_race_res_2025 = pd.read_csv('main_race_result(processed)/formula1_2025season_raceResults.csv')

main_race_quali_2022 = pd.read_csv('quali_both_result(processed)/race_quali_2022.csv')
main_race_quali_2023 = pd.read_csv('quali_both_result(processed)/race_quali_2023.csv')
main_race_quali_2024 = pd.read_csv('quali_both_result(processed)/race_quali_2024.csv')
main_race_quali_2025 = pd.read_csv('quali_both_result(processed)/race_quali_2025.csv')

sprint_quali_2023 = pd.read_csv('quali_both_result(processed)/sprint_quali_2023.csv')
sprint_quali_2024 = pd.read_csv('quali_both_result(processed)/sprint_quali_2024.csv')
sprint_quali_2025 = pd.read_csv('quali_both_result(processed)/sprint_quali_2025.csv')

sprint_race_res_2021 = pd.read_csv('sprint_race_result(processed)/sprint_2021_result.csv')
sprint_race_res_2022 = pd.read_csv('sprint_race_result(processed)/sprint_2022_result.csv')
sprint_race_res_2023 = pd.read_csv('sprint_race_result(processed)/sprint_2023_result.csv')
sprint_race_res_2024 = pd.read_csv('sprint_race_result(processed)/sprint_2024_result.csv')
sprint_race_res_2025 = pd.read_csv('sprint_race_result(processed)/sprint_2025_result.csv')

In [33]:
drivers_2019 = pd.read_csv("f12019season_drivers_processed.csv")
drivers_2020 = pd.read_csv("f12020season_drivers_processed.csv")
drivers_2021 = pd.read_csv("f12021season_drivers_processed.csv")
drivers_2022 = pd.read_csv("f12022season_drivers_processed.csv")
drivers_2023 = pd.read_csv("f12023season_drivers_processed.csv")
drivers_2024 = pd.read_csv("f12024season_drivers_processed.csv")
drivers_2025 = pd.read_csv("f12025season_drivers_processed.csv")

drivers_2019["year"] = 2019
drivers_2020["year"] = 2020
drivers_2021["year"] = 2021
drivers_2022["year"] = 2022
drivers_2023["year"] = 2023
drivers_2024["year"] = 2024
drivers_2025["year"] = 2025

team_master = pd.concat([
    drivers_2019,
    drivers_2020,
    drivers_2021,
    drivers_2022,
    drivers_2023,
    drivers_2024,
    drivers_2025
], ignore_index=True)

team_master = team_master[["kaggle_driver_id", "year", "Team"]]

In [34]:
def standardize_team(df, year):
    df = df.copy()

    # Add year column if missing
    if "year" not in df.columns:
        df["year"] = year

    # Drop existing Team if present
    if "Team" in df.columns:
        df = df.drop(columns=["Team"])

    # Merge standardized team
    df = df.merge(
        team_master,
        on=["kaggle_driver_id", "year"],
        how="left"
    )

    # Validation check
    missing = df["Team"].isna().sum()
    if missing > 0:
        print(f"Warning: {missing} team values missing for year {year}")

    return df

In [35]:
main_race_res_2019 = standardize_team(main_race_res_2019, 2019)
main_race_res_2020 = standardize_team(main_race_res_2020, 2020)
main_race_res_2021 = standardize_team(main_race_res_2021, 2021)
main_race_res_2022 = standardize_team(main_race_res_2022, 2022)
main_race_res_2023 = standardize_team(main_race_res_2023, 2023)
main_race_res_2024 = standardize_team(main_race_res_2024, 2024)
main_race_res_2025 = standardize_team(main_race_res_2025, 2025)
main_race_quali_2022 = standardize_team(main_race_quali_2022, 2022)
main_race_quali_2023 = standardize_team(main_race_quali_2023, 2023)
main_race_quali_2024 = standardize_team(main_race_quali_2024, 2024)
main_race_quali_2025 = standardize_team(main_race_quali_2025, 2025)
sprint_quali_2023 = standardize_team(sprint_quali_2023, 2023)
sprint_quali_2024 = standardize_team(sprint_quali_2024, 2024)
sprint_quali_2025 = standardize_team(sprint_quali_2025, 2025)

In [36]:
sprint_race_res_2021 = standardize_team(sprint_race_res_2021, 2021)
sprint_race_res_2022 = standardize_team(sprint_race_res_2022, 2022)
sprint_race_res_2023 = standardize_team(sprint_race_res_2023, 2023)
sprint_race_res_2024 = standardize_team(sprint_race_res_2024, 2024)
sprint_race_res_2025 = standardize_team(sprint_race_res_2025, 2025)

In [37]:
for df in [
    main_race_res_2019, main_race_res_2020, main_race_res_2021,
    main_race_res_2022, main_race_res_2023, main_race_res_2024, main_race_res_2025,
    main_race_quali_2022, main_race_quali_2023, main_race_quali_2024, main_race_quali_2025,
    sprint_quali_2023, sprint_quali_2024, sprint_quali_2025,
    sprint_race_res_2021, sprint_race_res_2022, sprint_race_res_2023,
    sprint_race_res_2024, sprint_race_res_2025
]:
    assert df["Team"].isna().sum() == 0

In [38]:
main_race_res_2019.to_csv('main_race_result(processed)/formula1_2019season_raceResults.csv', index=False)
main_race_res_2020.to_csv('main_race_result(processed)/formula1_2020season_raceResults.csv', index=False)
main_race_res_2021.to_csv('main_race_result(processed)/formula1_2021season_raceResults.csv', index=False)
main_race_res_2022.to_csv('main_race_result(processed)/formula1_2022season_raceResults.csv', index=False)
main_race_res_2023.to_csv('main_race_result(processed)/formula1_2023season_raceResults.csv', index=False)
main_race_res_2024.to_csv('main_race_result(processed)/formula1_2024season_raceResults.csv', index=False)
main_race_res_2025.to_csv('main_race_result(processed)/formula1_2025season_raceResults.csv', index=False)

main_race_quali_2022.to_csv('quali_both_result(processed)/race_quali_2022.csv', index=False)
main_race_quali_2023.to_csv('quali_both_result(processed)/race_quali_2023.csv', index=False)
main_race_quali_2024.to_csv('quali_both_result(processed)/race_quali_2024.csv', index=False)
main_race_quali_2025.to_csv('quali_both_result(processed)/race_quali_2025.csv', index=False)

sprint_quali_2023.to_csv('quali_both_result(processed)/sprint_quali_2023.csv', index=False)
sprint_quali_2024.to_csv('quali_both_result(processed)/sprint_quali_2024.csv', index=False)
sprint_quali_2025.to_csv('quali_both_result(processed)/sprint_quali_2025.csv', index=False)

sprint_race_res_2021.to_csv('sprint_race_result(processed)/sprint_2021_result.csv', index=False)
sprint_race_res_2022.to_csv('sprint_race_result(processed)/sprint_2022_result.csv', index=False)
sprint_race_res_2023.to_csv('sprint_race_result(processed)/sprint_2023_result.csv', index=False)
sprint_race_res_2024.to_csv('sprint_race_result(processed)/sprint_2024_result.csv', index=False)
sprint_race_res_2025.to_csv('sprint_race_result(processed)/sprint_2025_result.csv', index=False)

test = pd.read_csv('main_race_result(processed)/formula1_2023season_raceResults.csv')
print(test.head())

  Position  Starting Grid  kaggle_driver_id  raceId  finished  dnf  laps_down  \
0        1              1               830    1098       1.0  0.0        0.0   
1        2              2               815    1098       1.0  0.0        0.0   
2        3              5                 4    1098       1.0  0.0        0.0   
3        4              4               832    1098       1.0  0.0        0.0   
4        5              7                 1    1098       1.0  0.0        0.0   

   time_gap_sec  year             Team  
0         0.000  2023  Red Bull Racing  
1        11.987  2023  Red Bull Racing  
2        38.637  2023     Aston Martin  
3        48.052  2023          Ferrari  
4        50.977  2023         Mercedes  
